In [18]:
import os
import shutil
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

In [2]:
#path to store the csv
output_dir = "./localstorage/ecephys_cache_dir"


manifest_path = os.path.join(output_dir, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)


#getting all the tables
sessions_df = cache.get_session_table()
probes_df = cache.get_probes()
channels_df = cache.get_channels()
units_df = cache.get_units()

In [3]:
sessions_df.iloc[[24]]

,published_at,specimen_id,session_type,age_in_days,sex,full_genotype,unit_count,channel_count,probe_count,ecephys_structure_acronyms
id,,,,,,,,,,
762120172,2019-10-03T00:00:00Z,745276236,brain_observatory_1.1,100.0,M,Vip-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,717,1860,5,"[grey, VISal, nan, VISl, VISp, VIS, VISam]"


In [4]:
#finally got it to downloadd
session1 = cache.get_session_data(
    sessions_df.index.values[24],
    isi_violations_maximum = float('inf'),
    amplitude_cutoff_maximum = float('inf'),
    presence_ratio_minimum = -float('inf')
)

print(f"Session {sessions_df.index.values[24]} loaded successfully!")
print(f"Total units (neurons): {len(session1.units)}")

Session 762120172 loaded successfully!
Total units (neurons): 1642


In [5]:
drifting_table1 = session1.stimulus_presentations[session1.stimulus_presentations.stimulus_name == "drifting_gratings"]

len(drifting_table1)

630

In [6]:
drifting_table1

,stimulus_block,start_time,stop_time,contrast,x_position,color,orientation,y_position,frame,stimulus_name,size,spatial_frequency,phase,temporal_frequency,duration,stimulus_condition_id
stimulus_presentation_id,,,,,,,,,,,,,,,,
3798,2.0,1592.948984,1594.950644,0.8,null,null,225.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246
3799,2.0,1595.951487,1597.953167,0.8,null,null,135.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",2.0,2.00168,247
3800,2.0,1598.954004,1600.955644,0.8,null,null,225.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00164,248
3801,2.0,1601.956507,1603.958177,0.8,null,null,45.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",2.0,2.00167,249
3802,2.0,1604.959034,1606.960684,0.8,null,null,135.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",2.0,2.00165,247
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49428,7.0,5391.122617,5393.124297,0.8,null,null,45.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",8.0,2.00168,265
49429,7.0,5394.125144,5396.126794,0.8,null,null,270.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",8.0,2.00165,285
49430,7.0,5397.127637,5399.129317,0.8,null,null,180.0,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",2.0,2.00168,281


In [7]:
session_unit1 = session1.units[session1.units["ecephys_structure_acronym"].isin(["VISp"])
                               & (session1.units.isi_violations < 0.5)
                               & (session1.units.amplitude_cutoff < 0.1)
                               & (session1.units.presence_ratio > 0.9)]

spike_counts1 = session1.presentationwise_spike_counts(
    bin_edges=[0,2.0],
    stimulus_presentation_ids=drifting_table1.index,
    unit_ids=session_unit1.index
)

session_unit1

,waveform_PT_ratio,waveform_amplitude,amplitude_cutoff,cluster_id,cumulative_drift,d_prime,firing_rate,isi_violations,isolation_distance,L_ratio,...,ecephys_structure_id,ecephys_structure_acronym,anterior_posterior_ccf_coordinate,dorsal_ventral_ccf_coordinate,left_right_ccf_coordinate,probe_description,location,probe_sampling_rate,probe_lfp_sampling_rate,probe_has_lfp_data
unit_id,,,,,,,,,,,,,,,,,,,,,
951869857,0.599222,159.99672,0.002672,295,190.03,2.121158,0.837062,0.417065,39.174680,0.013506,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
951869927,0.461788,91.14807,0.039226,300,313.01,2.176517,3.038721,0.336253,33.318748,0.068869,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
951869889,0.372887,93.63939,0.026483,297,171.00,2.534168,5.747939,0.148153,55.853066,0.015258,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
951870013,0.387984,164.03283,0.001337,306,57.83,4.560852,1.347251,0.000000,68.793003,0.000504,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
951869958,0.504386,113.29227,0.074867,302,127.44,3.361614,0.380084,0.000000,23.934092,0.029283,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951872625,0.431735,173.48955,0.001993,470,132.58,5.214571,1.494996,0.081718,57.911138,0.002113,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
951872791,0.566744,107.32995,0.018432,479,199.34,3.658216,1.146196,0.222434,49.934805,0.008267,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True
951872885,0.354377,167.96442,0.000409,484,125.99,6.232315,2.649672,0.254942,79.584944,0.001523,...,385.0,VISp,NaN,NaN,NaN,probeC,See electrode locations,29999.995019,1249.999792,True


In [12]:
session_unit2 = session1.units[session1.units["ecephys_structure_acronym"].isin(["VISl"])
                               & (session1.units.isi_violations < 0.5)
                               & (session1.units.amplitude_cutoff < 0.1)
                               & (session1.units.presence_ratio > 0.9)]

spike_counts2 = session1.presentationwise_spike_counts(
    bin_edges=[0,2.0],
    stimulus_presentation_ids=drifting_table1.index,
    unit_ids=session_unit2.index
)

session_unit2

,waveform_PT_ratio,waveform_amplitude,amplitude_cutoff,cluster_id,cumulative_drift,d_prime,firing_rate,isi_violations,isolation_distance,L_ratio,...,ecephys_structure_id,ecephys_structure_acronym,anterior_posterior_ccf_coordinate,dorsal_ventral_ccf_coordinate,left_right_ccf_coordinate,probe_description,location,probe_sampling_rate,probe_lfp_sampling_rate,probe_has_lfp_data
unit_id,,,,,,,,,,,,,,,,,,,,,
951878084,0.124101,75.76296,0.007427,189,298.98,3.177852,2.943713,0.042154,81.936289,0.001238,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
951878034,0.597500,136.49259,0.010432,186,151.65,3.493603,12.137588,0.177037,83.235047,0.003519,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
951881384,0.059562,125.88693,0.014290,401,166.85,2.510913,0.272492,0.000000,42.728993,0.006199,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
951878139,0.922154,204.53043,0.033791,191,994.84,3.232700,1.143375,0.167650,60.727036,0.001069,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
951878053,0.439204,93.66435,0.001677,187,246.81,3.731279,4.932631,0.090079,53.925215,0.013285,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951880256,0.453896,117.73359,0.013685,322,136.16,4.290516,5.764726,0.178069,60.547331,0.006983,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
951881575,0.694626,169.75920,0.013647,414,118.93,4.164837,0.339056,0.317752,32.323027,0.037817,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True
951880483,0.687544,158.46831,0.042758,336,182.97,3.895513,2.776865,0.251071,47.775440,0.019652,...,409.0,VISl,NaN,NaN,NaN,probeD,See electrode locations,29999.921282,1249.99672,True


In [13]:
#adding the acronyms for Visp

spike_df1 = spike_counts1.to_dataframe().reset_index()

spike_df1 = spike_df1.merge(
    session_unit1[['ecephys_structure_acronym']], 
    left_on='unit_id', 
    right_index=True
)

spike_df1 = pd.merge(spike_df1, drifting_table1, left_on="stimulus_presentation_id", right_index=True)

spike_df1['firing_rate_hz'] = spike_df1['spike_counts'] / (spike_df1["stop_time"] - spike_df1["start_time"])

spike_df1

,stimulus_presentation_id,time_relative_to_stimulus_onset,unit_id,spike_counts,ecephys_structure_acronym,stimulus_block,start_time,stop_time,contrast,x_position,...,y_position,frame,stimulus_name,size,spatial_frequency,phase,temporal_frequency,duration,stimulus_condition_id,firing_rate_hz
0,3798,1.0,951869857,0,VISp,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,0.000000
1,3798,1.0,951869927,35,VISp,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,17.485487
2,3798,1.0,951869889,5,VISp,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,2.497927
3,3798,1.0,951870013,0,VISp,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,0.000000
4,3798,1.0,951869958,0,VISp,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55435,49432,1.0,951872625,0,VISp,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,0.000000
55436,49432,1.0,951872791,0,VISp,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,0.000000
55437,49432,1.0,951872885,3,VISp,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,1.498756
55438,49432,1.0,951873010,0,VISp,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,0.000000


In [14]:
#adding the acronyms for Visl

spike_df2 = spike_counts2.to_dataframe().reset_index()

spike_df2 = spike_df2.merge(
    session_unit2[['ecephys_structure_acronym']], 
    left_on='unit_id', 
    right_index=True
)

spike_df2 = pd.merge(spike_df2, drifting_table1, left_on="stimulus_presentation_id", right_index=True)

spike_df2['firing_rate_hz'] = spike_df2['spike_counts'] / (spike_df2["stop_time"] - spike_df2["start_time"])

spike_df2

,stimulus_presentation_id,time_relative_to_stimulus_onset,unit_id,spike_counts,ecephys_structure_acronym,stimulus_block,start_time,stop_time,contrast,x_position,...,y_position,frame,stimulus_name,size,spatial_frequency,phase,temporal_frequency,duration,stimulus_condition_id,firing_rate_hz
0,3798,1.0,951878084,4,VISl,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,1.998341
1,3798,1.0,951878034,26,VISl,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,12.989219
2,3798,1.0,951881384,0,VISl,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,0.000000
3,3798,1.0,951878139,8,VISl,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,3.996683
4,3798,1.0,951878053,1,VISl,2.0,1592.948984,1594.950644,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",1.0,2.00166,246,0.499585
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48505,49432,1.0,951880256,7,VISl,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,3.497097
48506,49432,1.0,951881575,0,VISl,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,0.000000
48507,49432,1.0,951880483,1,VISl,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,0.499585
48508,49432,1.0,951880581,1,VISl,7.0,5403.132657,5405.134317,0.8,null,...,null,null,drifting_gratings,"[250.0, 250.0]",0.04,"[79634.75, 79634.75]",15.0,2.00166,253,0.499585
